# What is LangSmith?

### One-line answer

> **LangSmith is a CCTV camera + flight recorder for your LLM app.**
> Every question, every document, every prompt, every reply - recorded, timed, and priced.

### How LangSmith maps to the 5 pillars of observability

| Pillar        | In LangSmith                                      |
| ------------- | ------------------------------------------------- |
| Logging       | Every run stores its exact **inputs and outputs** |
| Tracing       | The **trace tree** - its core feature             |
| Metrics       | Latency, tokens, **cost**, error rate             |
| User feedback | `client.create_feedback(...)` attached to a run   |
| Evaluation    | **Datasets + evaluators**                         |

---

build a RAG app


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langsmith import Client

client = Client()
print("Connected")

Connected


In [3]:
import time 
from langsmith import traceable

@traceable(run_sync="retriever")
def seach_docs(question):
    time.sleep(0.2)
    return ["refund_policy.md", "faq.md"]

@traceable(run_sync="llm")
def fake_llm(question, docs):
    time.sleep(0.4)
    return "Based on " + str(len(docs)) + " documents, here is the answer."

@traceable
def pipeline(question):
    docs = seach_docs(question)
    answer = fake_llm(question, docs)
    return answer

print(pipeline("What is the refund policy?"))

Based on 2 documents, here is the answer.


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name=os.environ.get("LLM_MODEL"),
    api_key=os.environ.get("LLM_API_KEY"),
    base_url=os.environ.get("LLM_BASE_URL"),
    temperature=0.1, 
)

result = llm.invoke("reply with exactly: langsmith is listening.")
print(result.content)

langsmith is listening.


In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

documents = [
    """
    REFUND ELIGIBILITY

    Customers may request a refund within 30 days of the purchase date.
    To be eligible for a refund, the product must be unused and in its
    original condition. Digital products may be eligible for a refund
    only if they have not been substantially used or downloaded.
    """,

    """
    NON-REFUNDABLE ITEMS

    The following items are non-refundable:
    - Gift cards
    - Discounted or clearance items
    - Personalized products
    - Products damaged by the customer
    - Services that have already been fully completed
    """,

    """
    HOW TO REQUEST A REFUND

    To request a refund, contact our customer support team with your
    order number, registered email address, and reason for the refund.
    Refund requests are typically reviewed within 3 to 5 business days.
    """,

    """
    REFUND PROCESSING TIME

    Once a refund is approved, the refund will be processed to the
    original payment method. Credit and debit card refunds may take
    5 to 10 business days to appear. Bank transfer refunds may take
    up to 7 business days.
    """,

    """
    SUBSCRIPTION REFUND POLICY

    Customers may cancel their subscription at any time. Monthly
    subscription payments are generally non-refundable after the
    billing period has started. Annual subscriptions may be eligible
    for a partial refund if cancelled within 14 days of renewal.
    """,

    """
    DAMAGED OR INCORRECT PRODUCTS

    If you receive a damaged, defective, or incorrect product, contact
    customer support within 7 days of delivery. You may be eligible for
    a replacement or a full refund. Supporting photographs may be
    required to process the request.
    """,

    """
    LATE REFUND REQUESTS

    Refund requests submitted after the standard 30-day refund period
    are normally not accepted. Exceptions may be considered for
    technical errors, duplicate charges, or other special circumstances.
    """
]

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = InMemoryVectorStore.from_texts(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

found = retriever.invoke("I need my money back, how long does it take?")

for doc in found:
    print("-", doc.page_content)

- 
    REFUND PROCESSING TIME

    Once a refund is approved, the refund will be processed to the
    original payment method. Credit and debit card refunds may take
    5 to 10 business days to appear. Bank transfer refunds may take
    up to 7 business days.
    
- 
    HOW TO REQUEST A REFUND

    To request a refund, contact our customer support team with your
    order number, registered email address, and reason for the refund.
    Refund requests are typically reviewed within 3 to 5 business days.
    


In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([    
    ("system", "You are a support assisstant Answer only using the context below. \n\n{context}"),
    ("human", "{question}"),
])

@traceable(name="rag_answer")
def rag_answer(question):
    docs = retriever.invoke(question)
    
    context = ""
    for doc in docs:
        context = context + "- " + doc.page_content + "\n"
    
    messages = prompt.format_messages(context=context, question=question)
    reply = llm.invoke(messages)
    
    return reply.content

In [7]:
print(rag_answer("How many days will it take to refund?"))

Refunds typically take:

- **Credit or debit card**: 5 to 10 business days to appear on your statement.  
- **Bank transfer**: up to 7 business days to reach your account.  

(These times apply after the refund has been approved and processed.)


# Agent Observability

In [8]:
from langchain_core.tools import tool

@tool
def get_order_status(order_id: str) -> str:
    """ Look up the delivery status of an order_id by its ID. """
    orders = {"A123": "Shipped, arriving Tuesday", "B4567": "Processing"}
    if order_id in orders:
        return orders[order_id]
    return "Order not found."

@tool
def calculate_refund(price: float, days_since_purchases) -> str:
    """ Calculate the refund amount for an order. """
    if days_since_purchases > 30:
        return "Not eligible: Purchased more than days ago."
    return "Eligible for full refund of " + str(price)

tools = [calculate_refund, get_order_status]

In [9]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

llm_with_tools = llm.bind_tools(tools)

def call_model(state):
    """ The brain: Look at the conversation so far and decide what to do  """
    reply = llm_with_tools.invoke(state["messages"])
    return {"messages": [reply]}

builder = StateGraph(MessagesState)

builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

agent = builder.compile()

In [10]:
question = "My order A123 cost of 500 and i bought it 10 day before. where is it, and what refund i would get?"
final_state = agent.invoke({"messages": [("human", question)]})


for message in final_state["messages"]:
    message.pretty_print()

================================ Human Message =================================

My order A123 cost of 500 and i bought it 10 day before. where is it, and what refund i would get?
================================== Ai Message ==================================
Tool Calls:
  get_order_status (chatcmpl-tool-5e16f8ea386e4610a0868fc8f838ef21)
 Call ID: chatcmpl-tool-5e16f8ea386e4610a0868fc8f838ef21
  Args:
    order_id: A123
================================= Tool Message =================================
Name: get_order_status

Shipped, arriving Tuesday
================================== Ai Message ==================================
Tool Calls:
  calculate_refund (chatcmpl-tool-0c958a58910d496cbb5686b7354fe8e7)
 Call ID: chatcmpl-tool-0c958a58910d496cbb5686b7354fe8e7
  Args:
    price: 500
    days_since_purchases: 10
================================= Tool Message =================================
Name: calculate_refund

Eligible for full refund of 500.0
==================================